In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [2]:
# Example sentence (decoder-only context)
sentence = ["ChatGPT", "is", "amazing", "."]
seq_len = len(sentence)
vocab_size = 10000
embedding_dim = 16
num_heads = 2
hidden_dim = 32

In [3]:
# 1️⃣ Token embeddings
token_embedding = nn.Embedding(vocab_size, embedding_dim)
token_ids = torch.arange(seq_len)
tokens_emb = token_embedding(token_ids)  # [seq_len, embedding_dim]

In [4]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

pos_encoder = PositionalEncoding(embedding_dim)
tokens_emb = pos_encoder(tokens_emb)  # [seq_len, embedding_dim]

In [5]:
# 3️⃣ Decoder-only Transformer layer (masked self-attention)
decoder_layer = nn.TransformerDecoderLayer(
    d_model=embedding_dim,
    nhead=num_heads,
    dim_feedforward=hidden_dim
)
# Wrap in nn.TransformerDecoder with dummy memory (not used)
transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=1)

# Add batch dimension: [seq_len, batch_size, embedding_dim]
tokens_emb = tokens_emb.unsqueeze(1)

# Create causal mask to prevent attending to future tokens
tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len)

In [6]:
# Forward pass (memory=None because decoder-only)
# We pass an empty memory tensor to satisfy nn.TransformerDecoder API
memory = torch.zeros(0, 1, embedding_dim)  # [0, batch_size, d_model]
output = transformer_decoder(tgt=tokens_emb, memory=memory, tgt_mask=tgt_mask)

print("Decoder-only output shape:", output.shape)
print("Decoder-only output:\n", output.squeeze(1))

Decoder-only output shape: torch.Size([4, 1, 16])
Decoder-only output:
 tensor([[-1.9994e+00, -6.0862e-04,  2.8717e-01,  1.5250e-01,  1.3168e+00,
          4.1017e-01, -1.7818e+00,  1.2921e+00,  1.6685e-01,  9.7566e-01,
         -1.1651e+00, -3.5378e-01, -9.0014e-01,  1.0646e+00,  8.1564e-01,
         -2.8070e-01],
        [-8.1184e-01,  1.4251e+00, -1.8396e+00,  1.2122e+00, -1.6961e+00,
          8.0022e-02, -3.7921e-01, -2.0908e-02,  2.5834e-01,  1.1982e+00,
         -5.4901e-01,  1.4758e+00, -7.4829e-01, -3.3322e-01, -1.4130e-01,
          8.6980e-01],
        [-3.8454e-01, -3.1430e+00,  5.4068e-01,  9.7700e-01,  8.5845e-01,
          3.5499e-01,  5.2761e-01,  7.7208e-01,  1.0828e+00,  1.3841e-01,
         -4.5982e-01,  3.4796e-01, -5.4543e-02, -5.1346e-01, -1.0935e+00,
          4.8916e-02],
        [ 6.9176e-01, -9.2705e-01,  8.0257e-01,  1.3862e+00,  1.6732e-01,
         -1.8532e+00, -1.1129e+00, -1.1408e+00, -1.2903e+00,  5.8024e-01,
         -1.0300e-01,  1.8135e+00,  5.2290e-0

example to show autoregressive generation:

In [7]:
max_len = 5  # generate up to 5 tokens

In [8]:
# 4️⃣ Autoregressive generation loop
generated_tokens = [torch.tensor([0])]  # start token, shape [1]

for step in range(max_len):
    # Prepare input sequence
    input_ids = torch.cat(generated_tokens)  # shape [seq_len]
    token_emb = token_embedding(input_ids)   # [seq_len, embedding_dim]
    token_emb = pos_encoder(token_emb)
    token_emb = token_emb.unsqueeze(1)       # [seq_len, batch_size=1, embedding_dim]

    # Causal mask
    seq_len = token_emb.size(0)
    tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len)

    # Forward pass (memory is empty for decoder-only)
    memory = torch.zeros(0, 1, embedding_dim)
    output = transformer_decoder(tgt=token_emb, memory=memory, tgt_mask=tgt_mask)

    # Map last token embedding to logits (toy linear layer)
    linear = nn.Linear(embedding_dim, vocab_size)
    logits = linear(output[-1, 0])
    probs = F.softmax(logits, dim=-1)

    # Sample next token
    next_token = torch.multinomial(probs, num_samples=1)  # shape [1]
    generated_tokens.append(next_token)

# Convert to indices
generated_indices = [t.item() for t in generated_tokens]
print("Generated token indices:", generated_indices)

Generated token indices: [0, 7959, 4569, 8951, 4435, 8804]


example for generation with encoder-decoder structure:

In [9]:
tgt_sentence = ["It", "is", "amazing", "."]
src_sentence = ["ChatGPT", "explains", "things", "clearly", "."]

tgt_len = len(tgt_sentence)
src_len = len(src_sentence)

tgt_ids = torch.arange(tgt_len)
src_ids = torch.arange(src_len)

tgt_emb = token_embedding(tgt_ids)  # [tgt_len, embedding_dim]
src_emb = token_embedding(src_ids)  # [src_len, embedding_dim]

tgt_emb = pos_encoder(tgt_emb)  # [tgt_len, embedding_dim]
src_emb = pos_encoder(src_emb)  # [src_len, embedding_dim]

tgt_emb = tgt_emb.unsqueeze(1)  # [tgt_len, 1, embedding_dim]
src_emb = src_emb.unsqueeze(1)  # [src_len, 1, embedding_dim]

tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len)

In [10]:
output = transformer_decoder(
    tgt=tgt_emb,
    memory=src_emb,
    tgt_mask=tgt_mask
)

print("Decoder output shape:", output.shape)
print("Decoder output:\n", output.squeeze(1))

Decoder output shape: torch.Size([4, 1, 16])
Decoder output:
 tensor([[-2.0681,  0.3329,  0.5039,  0.5057,  1.4950, -0.2135, -1.5852,  1.8777,
          0.1846,  0.2189, -0.7026, -0.3048, -0.4606,  0.2545,  0.9392, -0.9775],
        [-0.6768,  1.1531, -2.1982,  0.8672, -1.1739, -0.0642, -0.1617,  0.2557,
          0.8990,  1.2831, -0.5711,  1.2909, -0.3858, -1.2005, -0.3232,  1.0063],
        [-0.2393, -3.2336,  0.3995,  0.5453,  1.0209,  0.6488,  0.0274,  0.5811,
          0.3700,  0.8515, -0.3737,  0.0119,  0.4649, -0.3389, -1.2691,  0.5333],
        [ 0.7612, -1.0110,  0.7536,  1.2019,  0.7942, -1.5938, -1.9314, -1.1042,
         -1.0000,  1.0496, -0.1049,  0.8265,  0.8955, -0.4762,  0.4791,  0.4598]],
       grad_fn=<SqueezeBackward1>)
